In [ ]:
import sys
import os
# Add the src directory to Python path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Import the modules
import clathrate_analysis
import truncation_analysis
import tomogram_utils

# Reload the modules
import importlib
importlib.reload(clathrate_analysis)
importlib.reload(truncation_analysis)
importlib.reload(tomogram_utils)

# Now import the package
from clathrate_analysis import *#parse_particles_and_shape, plot_clathrate_cavities,  get_truncated_bipyramid_geometry,
from truncation_analysis import *#plot_truncation_cavity_comparison
from tomogram_utils import *#create_tomogram_from_particles

In [ ]:
# Load the file
# filename = 'C:\\Users\\b304014\\Software\\blee\\models\\Cages\\Cage A new.pos'
filename = 'C:\\Users\\b304014\\Software\\blee\\models\\ClaS_bipyramid_averaged.pos'
# filename="C:\\Users\\b304014\\Box\\zhihua\\models\\Right bipyramids ClaIV_cubic_bipyra_UC.pos"

# Parse particles and shape data
particles, shape_vertices, shape_color, simulation_data = parse_particles_and_shape(filename)

# Set truncation factor
truncation_factor = 0.0 # Can be adjusted between 0.0 and 1.0

# Improved cavity detection parameters
grid_size = 64  # Increased from 64 for better resolution
# min_cavity_size = 4000  # Reduced to detect smaller cavities
padding = 0.15  # Moderate padding for better cavity detection

# # For 2x2x2 duplication
# grid_size = 128*2
# min_cavity_size = 100 # Number of voxels for a cavity to be considered

# print(f"Minimum cavity size: {min_cavity_size}")
print(f"Grid size: {grid_size}")
print(f"Truncation factor: {truncation_factor}")
print(f"Padding: {padding}")

# Get truncated shape faces for tomogram creation
_, shape_faces, _ = get_truncated_bipyramid_geometry(
    shape_vertices, 
    shape_color,
    truncation_factor=truncation_factor
)

# Create nxnxn duplicated particles
duplicated_particles = duplicate_unit_cell(particles, nx=2, ny=2, nz=2, simulation_data=simulation_data)
duplicated_positions = [(pos, quat) for pos, quat, _ in duplicated_particles]


In [ ]:
filename,voxel_grid = create_tomogram_from_particles(duplicated_positions, grid_size=grid_size, padding=padding, shape_vertices=shape_vertices, shape_faces=shape_faces, 
                                 pixel_size=1.0, filename=None)

In [ ]:
plot_3d_tomogram('C:\\Users\\b304014\\Software\\blee\\models\\Clathrate_Analysis\\notebooks\\tomogram_20250828_131622.tif', plot_type='isosurface', threshold=0.8, opacity=0.7, colorscale='Viridis')

In [ ]:
# # Plot the duplicated truncated particles with cavities
# print(f"\nPlotting duplicated particles with truncation factor {truncation_factor} and cavities...")
# cavity_centers, cavity_volumes, cavity_radii = plot_clathrate_cavities(
#     particles=duplicated_positions,
#     shape_vertices=shape_vertices, 
#     shape_color=shape_color,
#     geometry_func=get_truncated_bipyramid_geometry,
#     truncation_factor=truncation_factor,
#     show_particles=True,
#     show_cavities=True,
#     show_spheres=True,
#     grid_size=grid_size,
#     padding=padding,  # Use the improved padding value
#     simulation_data=simulation_data,
#     min_cavity_size=min_cavity_size,
#     keep_largest_cavity_only=False  # Show all cavities first
# )

In [ ]:
cavities = detect_simple_cavities(
    particles=duplicated_positions, 
    shape_vertices=shape_vertices, 
    shape_color=shape_color,
    grid_size=128, 
    padding=0.2,
    min_radius=0.16,#0.15,#0.2,#0.33, 
    min_separation=0.2,  
    geometry_func=get_truncated_bipyramid_geometry, 
    truncation_factor=truncation_factor,
    min_surrounding_particles=2,
    max_empty_neighbors_fraction=0.8,
    boundary_margin=0.15
)
# Extract centers, radii and volumes into separate lists
cavity_centers = [cavity['center'] for cavity in cavities]
cavity_radii = [cavity['radius'] for cavity in cavities]
cavity_volumes = [cavity['volume'] for cavity in cavities]

print("Cavity centers:", cavity_centers)
print("Cavity radii:", cavity_radii)
print("Cavity volumes:", cavity_volumes)

In [ ]:
# After detecting cavities with detect_clathrate_cavities or find_central_cavity:
tomogram, edges = plot_cavity_objects(
    particles=duplicated_positions,
    cavity_centers=cavity_centers,
    cavity_radii=cavity_radii,
    shape_vertices=shape_vertices,
    shape_color=shape_color,
    geometry_func=get_truncated_bipyramid_geometry, 
    show_particles=True,  # Set to False to hide original particles
    simulation_data=simulation_data,
    cavity_object_type='cube',
    truncation_factor=truncation_factor
)

In [ ]:
plot_truncated_particle(
    pos=(0, 0, 0),  # center position
    quat=(1, 0, 0, 0),  # no rotation
    shape_vertices=shape_vertices,
    shape_color='gray',#shape_color,
    truncation_factor=0.0,#truncation_factor,  # or 0.0 for non-truncated bipyramid
    plot_type='surface',  # 'surface', 'wireframe', or 'both'
    show_axes=True
)

In [ ]:
tomogram_file = create_tomogram_with_cavity_objects(duplicated_positions, cavity_centers, cavity_radii, 
                                  grid_size=128, padding=0.1, shape_vertices=shape_vertices, shape_faces=shape_faces,
                                  pixel_size=1.0, filename=None, geometry_func=get_truncated_bipyramid_geometry, truncation_factor=truncation_factor,
                                  cavity_object_type='cube', cavity_object_scale=1.0)

# # Optionally visualize the tomogram
# plot_3d_tomogram(tomogram_file, plot_type='isosurface', threshold=0.75)

In [ ]:
# create_tomogram_from_particles(duplicated_positions, grid_size=128, 
#                                padding=0.1, shape_vertices=shape_vertices, 
#                                shape_faces=shape_faces, 
#                                  pixel_size=1.0, filename=None)

tomogram_file = create_tomogram_with_cavity_objects(duplicated_positions, cavity_centers=None, cavity_radii=None, 
                                  grid_size=128, padding=0.1, shape_vertices=shape_vertices, shape_faces=shape_faces,
                                  pixel_size=1.0, filename=None, geometry_func=get_truncated_bipyramid_geometry, truncation_factor=truncation_factor, cavity_object_scale=1.0)


In [ ]:
plot_saxs_comparison_cavities_cubes(base_path='C:\\Users\\b304014\\Software\\blee\\models\\SAXS\\3x3_clathrate_files\\', truncation_factor=truncation_factor)

plot_saxs_comparison(base_path='C:\\Users\\b304014\\Software\\blee\\models\\SAXS\\')

In [ ]:
# Place cubes in cavities (original behavior)
create_tomogram_with_cavity_objects(
    particles=duplicated_positions,
    cavity_centers=cavity_centers,
    cavity_radii=cavity_radii,
    shape_vertices=shape_vertices,
    cavity_object_type='cube'
)


In [ ]:

# Place bipyramids in cavities
create_tomogram_with_cavity_objects(
    particles=duplicated_positions,
    cavity_centers=cavity_centers,
    cavity_radii=cavity_radii,
    shape_vertices=shape_vertices,
    cavity_object_type='bipyramid',
    cavity_object_scale=2.0  # Adjust size as needed
)

In [ ]:

# Calculate 3D FFT of the tomogram
fft_data = plot_fft_magnitude(tomogram,edges,threshold=0.75)
